# Simulatie tariefspiraal

Deze notebook simuleert de adverse-selectiespiraal op de Vlaamse elektriciteitsmarkt: wanneer goedkope klanten overstappen naar een dynamisch contract, stijgt het vaste tarief voor de achterblijvers.

**Volgorde van uitvoeren:**

1. **Ember-prijsdata voorbereiden** — extraheer de Belgische day-ahead prijzen voor 2022 en 2024
2. **Fluvius kwartierdata aggregeren naar uurdata** — voor elk van de 8 profieltypes
3. **Steekproef trekken** — 1.000 huishoudens per scenario op basis van penetratiegraden
4. **Hoofdsimulatie** — tariefberekening, death spiral, gevoeligheidsanalyse

**Inputbestanden:**

- `Belgium.csv` (Ember European Wholesale Electricity Price Data)
- `P6269_Open_Data_*.csv` (8 Fluvius-profielbestanden, kwartierdata 2024)

**Belangrijke parameters:**

- Risicopremie vast contract: 5%
- Dynamische tariefformule: Eneco Zon & Wind Dynamisch, januari 2024
- Random seed: 42 (reproduceerbaarheid)

## Stap 1 — Ember-prijsdata voorbereiden

Extraheert de Belgische day-ahead groothandelsprijzen uit de volledige Ember-dataset voor de gewenste jaren. De prijzen worden bewaard in EUR/MWh.

- **2024** = referentiejaar (gemiddelde 70,3 EUR/MWh, std 43,0 EUR/MWh)
- **2022** = crisisscenario (mediaan 244,5 EUR/MWh, std 134,7 EUR/MWh)

In [ ]:
import pandas as pd

# Parameters
BESTANDSPAD = "Belgium.csv"
JAREN       = [2022, 2024]

# Laad de volledige Ember-dataset
df = pd.read_csv(BESTANDSPAD)
df.columns = ["Country", "ISO3 Code", "Datetime (UTC)", "Datetime (Local)", "Price (EUR/MWh)"]
df["Datetime (Local)"] = pd.to_datetime(df["Datetime (Local)"])

# Maak per jaar een apart bestand
for jaar in JAREN:
    subset = df[df["Datetime (Local)"].dt.year == jaar].copy()
    subset = subset[["Datetime (Local)", "Price (EUR/MWh)"]].reset_index(drop=True)
    uitvoer = f"Belgium_{jaar}_MWh.csv"
    subset.to_csv(uitvoer, index=False)

    print(f"{jaar}: {len(subset)} uren → {uitvoer}")
    print(f"  Gemiddelde : {subset['Price (EUR/MWh)'].mean():.1f} EUR/MWh")
    print(f"  Standaardafw.: {subset['Price (EUR/MWh)'].std():.1f} EUR/MWh")
    print(f"  Negatieve uren: {(subset['Price (EUR/MWh)'] < 0).sum()} "
          f"({(subset['Price (EUR/MWh)'] < 0).mean()*100:.1f}%)\n")

## Stap 2 — Fluvius kwartierdata aggregeren naar uurdata

De ruwe Fluvius open data bevat metingen per kwartier (vier metingen per uur). Voor de simulatie hebben we uurwaarden nodig, omdat de groothandelsprijzen ook uurlijks zijn.

**Werking:** Voor elk uur worden de vier kwartiervolumes opgeteld. De technologie-indicatoren (PV, WP, EV) zijn constant per huishouden en worden overgenomen.

Dit moet één keer per profielbestand worden uitgevoerd. Onderstaande cel verwerkt alle 8 profieltypes in één keer.

In [ ]:
import pandas as pd

# Koppeling tussen Fluvius open-data bestanden en interne profielnamen
PROFIELEN = {
    "P6269_Open_Data_geen_ZP.csv":        "fluvius_300_zonder_middelen_uur.csv",
    "P6269_Open_Data_enkel_ZP.csv":       "fluvius_300_met_ZP_uur.csv",
    "P6269_Open_Data_WP_geen_ZP.csv":     "fluvius_300_met_WP_uur.csv",
    "P6269_Open_Data_EV_geen_ZP.csv":     "fluvius_300_met_EV_uur.csv",
    "P6269_Open_Data_WP_met_ZP.csv":      "fluvius_300_met_WP_met_ZP_uur.csv",
    "P6269_Open_Data_EV_met_ZP.csv":      "fluvius_300_met_EV_met_ZP_uur.csv",
    "P6269_Open_Data_WP_EV_geen_ZP.csv":  "fluvius_300_met_WP_met_EV_uur.csv",
    "P6269_Open_Data_WP_EV_met_ZP.csv":   "fluvius_300_met_WP_met_EV_met_ZP_uur.csv",
}


def kwartier_naar_uur(input_bestand, output_bestand):
    """Aggregeer 15-minutendata van één Fluvius-bestand naar uurdata."""
    df = pd.read_csv(input_bestand)

    # Tijdstempel in UTC inlezen (herkenbaar aan Z-suffix in de data)
    # De aggregatie gebeurt in UTC zodat de zomertijdovergang geen problemen geeft
    df["Datum_Startuur"] = pd.to_datetime(df["Datum_Startuur"], utc=True)
    df["Uur"] = df["Datum_Startuur"].dt.floor("h")

    # Tel volumes per uur op; technologie-indicatoren zijn constant per klant
    df_uur = df.groupby(["EAN_ID", "Datum", "Uur"]).agg(
        Volume_Afname_KWh             = ("Volume_Afname_KWh",             "sum"),
        Volume_Injectie_KWh           = ("Volume_Injectie_KWh",           "sum"),
        Warmtepomp_Indicator          = ("Warmtepomp_Indicator",          "first"),
        Elektrisch_Voertuig_Indicator = ("Elektrisch_Voertuig_Indicator", "first"),
        PV_Installatie_Indicator      = ("PV_Installatie_Indicator",      "first"),
        Contract_Categorie            = ("Contract_Categorie",            "first"),
    ).reset_index().rename(columns={"Uur": "Datum_Startuur"})

    df_uur.to_csv(output_bestand, index=False)
    print(f"  {input_bestand} → {output_bestand} "
          f"({df['EAN_ID'].nunique()} klanten, {len(df_uur):,} uren)")


# Verwerk alle 8 profielbestanden
print("Aggregatie kwartier → uur:")
for input_bestand, output_bestand in PROFIELEN.items():
    kwartier_naar_uur(input_bestand, output_bestand)
print("\nAlle profielbestanden geaggregeerd.")

## Stap 3 — Steekproef van 1.000 huishoudens per scenario

Trekt voor elk van de vier scenario's een representatieve steekproef van 1.000 huishoudens, gewogen volgens de Vlaamse penetratiegraden voor PV, WP en EV.

**Vier scenario's:**

| Scenario | PV | WP | EV |
|---|---|---|---|
| 2024 (baseline) | 31% | 3% | 10% |
| 2034 | 50% | 15% | 45% |
| High EV | 50% | 15% | 80% |
| Volledige elektrificatie | 80% | 60% | 85% |

**Methodologie:**

- De kans op elk profieltype = product van afzonderlijke penetratiegraden (onafhankelijkheidsaanname)
- De aantallen worden bepaald via de *largest remainder method* (som = exact 1.000)
- Bij meer huishoudens nodig dan beschikbaar (300 per profiel) wordt met teruglegging getrokken
- Reproduceerbaar via vaste random seed (42)

In [ ]:
import pandas as pd
import numpy as np

RANDOM_SEED = 42
TOTAAL      = 1_000

# Koppeling tussen interne profielnamen en uurbestanden (output van stap 2)
BESTANDEN = {
    "zonder_middelen":      "fluvius_300_zonder_middelen_uur.csv",
    "met_ZP":               "fluvius_300_met_ZP_uur.csv",
    "met_WP":               "fluvius_300_met_WP_uur.csv",
    "met_EV":               "fluvius_300_met_EV_uur.csv",
    "met_WP_met_ZP":        "fluvius_300_met_WP_met_ZP_uur.csv",
    "met_EV_met_ZP":        "fluvius_300_met_EV_met_ZP_uur.csv",
    "met_WP_met_EV":        "fluvius_300_met_WP_met_EV_uur.csv",
    "met_WP_met_EV_met_ZP": "fluvius_300_met_WP_met_EV_met_ZP_uur.csv",
}

# Vier scenario's: penetratiegraden + outputbestand
SCENARIOS = {
    "Baseline 2024":            {"ZP": 0.31, "WP": 0.03, "EV": 0.10,
                                 "output": "dataset_1000_huishoudens.csv"},
    "2034":                     {"ZP": 0.50, "WP": 0.15, "EV": 0.45,
                                 "output": "dataset_1000_huishoudens_2034.csv"},
    "High EV":                  {"ZP": 0.50, "WP": 0.15, "EV": 0.80,
                                 "output": "dataset_1000_huishoudens_2034_highEV.csv"},
    "Volledige elektrificatie": {"ZP": 0.80, "WP": 0.60, "EV": 0.85,
                                 "output": "dataset_1000_huishoudens_volledige_elektrificatie.csv"},
}


def bereken_aantallen(ZP, WP, EV, totaal):
    """
    Bereken hoeveel huishoudens per profieltype nodig zijn.
    Onafhankelijkheidsaanname: P(combinatie) = product van afzonderlijke kansen.
    Resterende plaatsen worden toegekend via de largest remainder method.
    """
    gewichten = {
        "zonder_middelen":      (1-ZP) * (1-WP) * (1-EV),
        "met_ZP":               ZP     * (1-WP) * (1-EV),
        "met_WP":               (1-ZP) * WP     * (1-EV),
        "met_EV":               (1-ZP) * (1-WP) * EV,
        "met_WP_met_ZP":        ZP     * WP     * (1-EV),
        "met_EV_met_ZP":        ZP     * (1-WP) * EV,
        "met_WP_met_EV":        (1-ZP) * WP     * EV,
        "met_WP_met_EV_met_ZP": ZP     * WP     * EV,
    }
    # Exacte (niet-gehele) aantallen
    exact = {p: g * totaal for p, g in gewichten.items()}
    # Naar beneden afronden
    aantallen = {p: int(v) for p, v in exact.items()}
    # Resterende plaatsen verdelen aan profielen met de grootste fractionele rest
    verschil = totaal - sum(aantallen.values())
    voor_extra = sorted(exact, key=lambda p: exact[p] % 1, reverse=True)[:verschil]
    for profiel in voor_extra:
        aantallen[profiel] += 1
    return aantallen


def maak_steekproef(scenario_naam, ZP, WP, EV, output_bestand):
    """Trek een steekproef van 1.000 huishoudens voor één scenario."""
    print(f"\n── {scenario_naam} (ZP={ZP*100:.0f}%, WP={WP*100:.0f}%, EV={EV*100:.0f}%) ──")
    aantallen = bereken_aantallen(ZP, WP, EV, TOTAAL)

    rng = np.random.default_rng(RANDOM_SEED)
    delen = []

    for profiel, aantal in aantallen.items():
        if aantal == 0:
            continue

        df = pd.read_csv(BESTANDEN[profiel])
        eans_per_klant = {ean: groep.copy() for ean, groep in df.groupby("EAN_ID")}
        beschikbaar = list(eans_per_klant.keys())

        # Teruglegging als meer nodig dan beschikbaar (300 unieke meters per profiel)
        replace = aantal > len(beschikbaar)
        gekozen = rng.choice(beschikbaar, size=aantal, replace=replace)

        # Geef elke trekking een uniek ID (ook bij teruglegging)
        for i, ean in enumerate(gekozen):
            rijen = eans_per_klant[ean].copy()
            rijen["EAN_ID"]        = f"{profiel}_sample_{i}_orig_{int(ean)}"
            rijen["Origineel_EAN"] = int(ean)
            rijen["Profiel"]       = profiel
            delen.append(rijen)

        print(f"  {profiel:<25} {aantal:>4} ({aantal/TOTAAL*100:.1f}%)")

    # Voeg alles samen en hernoem klant-IDs naar gehele getallen 1..1000
    dataset = pd.concat(delen, ignore_index=True)
    mapping = {ean: i + 1 for i, ean in enumerate(dataset["EAN_ID"].unique())}
    dataset["EAN_ID"] = dataset["EAN_ID"].map(mapping)

    dataset.to_csv(output_bestand, index=False)
    print(f"  → {output_bestand} ({dataset['EAN_ID'].nunique()} klanten, {len(dataset):,} rijen)")


# Voer voor alle vier de scenario's de steekproeftrekking uit
print("STEEKPROEFTREKKING - VIER SCENARIO'S")
for naam, config in SCENARIOS.items():
    maak_steekproef(naam, config["ZP"], config["WP"], config["EV"], config["output"])
print("\nAlle steekproeven aangemaakt.")

## Stap 4 — Hoofdsimulatie

Deze stap voert de eigenlijke analyse uit en bestaat uit:

1. **Kostenpositie van alle 2.400 meters** — gelijk gewogen, om de heterogeniteit per profieltype te tonen (puntenwolk + kwartielverdeling).
2. **Tarieven per scenario** — vast portfoliotarief, individueel klanttarief, kost op vast vs dynamisch contract.
3. **Death spiral simulatie** — laat de goedkoopste klanten stap voor stap vertrekken en bereken het nieuwe portfoliogemiddelde.
4. **Crisisscenario 2022** — dezelfde klantenmix als baseline, maar gekoppeld aan de 2022-prijzen.
5. **Gevoeligheidsanalyse volatiliteit** — schaalfactoren f ∈ {1,0; 1,5; 2,0; 3,0} op de prijsafwijkingen.

Alle berekeningen gebeuren vanuit leveranciersperspectief, exclusief btw.

In [ ]:
import pandas as pd
import numpy as np

# ── Configuratie ─────────────────────────────────────────────

# Risicopremie die de leverancier toepast op een vast contract
RISICO_PREMIE = 0.05

# Tariefformule Eneco Zon & Wind Dynamisch (januari 2024)
# Afnameprijs per uur (€/kWh, excl. btw) = (DYN_ALPHA × EPEX_MWh + DYN_BETA) / 100
DYN_ALPHA, DYN_BETA = 0.102, 1.0
# Injectievergoeding per uur (€/kWh, excl. btw) = (DYN_ALPHA_INJ × EPEX_MWh − DYN_BETA_INJ) / 100
# Kan negatief worden bij lage EPEX-prijzen
DYN_ALPHA_INJ, DYN_BETA_INJ = 0.100, 1.188

# Death spiral simulatie: aantal stappen en drempel voor profielmarkers
SPIRAL_STAPPEN  = 50      # van 0% tot 98% in stappen van 2%
DREMPEL_PROFIEL = 0.80    # stippellijn bij 80% vertrek per profiel

# Gevoeligheidsanalyse: schaalfactoren op de prijsafwijking t.o.v. het jaargemiddelde
# Factor 3.0 is vergelijkbaar met de volatiliteit tijdens de energiecrisis 2022
VOLATILITEIT_SCENARIOS = {
    "Prijzen 2024 (referentie)":       1.0,
    "Prijzen 2024 +50% volatiliteit":  1.5,
    "Prijzen 2024 +100% volatiliteit": 2.0,
    "Prijzen 2024 +200% volatiliteit": 3.0,
}

# Bestandsnamen 2.400-meters dataset (kwartielanalyse)
FLUVIUS_2400 = {
    "Geen ZP/WP/EV": "fluvius_300_zonder_middelen_uur.csv",
    "ZP":            "fluvius_300_met_ZP_uur.csv",
    "WP":            "fluvius_300_met_WP_uur.csv",
    "EV":            "fluvius_300_met_EV_uur.csv",
    "ZP + WP":       "fluvius_300_met_WP_met_ZP_uur.csv",
    "ZP + EV":       "fluvius_300_met_EV_met_ZP_uur.csv",
    "WP + EV":       "fluvius_300_met_WP_met_EV_uur.csv",
    "ZP + WP + EV":  "fluvius_300_met_WP_met_EV_met_ZP_uur.csv",
}

# Bestandsnamen 1.000-steekproef per scenario
FLUVIUS_BESTANDEN = {
    "baseline":       "dataset_1000_huishoudens.csv",
    "2034":           "dataset_1000_huishoudens_2034.csv",
    "highev":         "dataset_1000_huishoudens_2034_highEV.csv",
    "elektrificatie": "dataset_1000_huishoudens_volledige_elektrificatie.csv",
}
SCENARIO_LABELS = {
    "baseline":       "Baseline 2024",
    "2034":           "2034",
    "highev":         "High EV",
    "elektrificatie": "Volledige elektrificatie",
}

# Vertaling profielnaam → leesbaar label
PROFIEL_LABELS = {
    "zonder_middelen":      "Geen ZP/WP/EV",
    "met_EV":               "EV",
    "met_WP":               "WP",
    "met_WP_met_EV":        "WP + EV",
    "met_ZP":               "ZP",
    "met_EV_met_ZP":        "ZP + EV",
    "met_WP_met_ZP":        "ZP + WP",
    "met_WP_met_EV_met_ZP": "ZP + WP + EV",
}
PROFIEL_VOLGORDE = [
    "Geen ZP/WP/EV", "EV", "WP", "WP + EV",
    "ZP", "ZP + EV", "ZP + WP", "ZP + WP + EV",
]
KWARTIELEN = ["Goedkoopste 25%", "25%-50%", "50%-75%", "Duurste 25%"]

### Hulpfuncties

Hieronder de hulpfuncties die in de hoofdsimulatie worden aangeroepen. Ze zijn opgedeeld in vier blokken:

- **`laad_epex`** — leest een Ember-prijsbestand in en behandelt zomertijd-/wintertijdovergangen.
- **`bereken_kostenpositie_2400`** — berekent het individuele leverancierstarief per klant voor alle 2.400 Fluvius-meters en deelt ze in kwartielen.
- **`laad_fluvius`** — leest een 1.000-steekproef en koppelt de EPEX-prijzen op maand/dag/uur.
- **`bereken_tarieven`** — berekent voor één scenario het vaste portfoliotarief en de kosten per klant op vast vs dynamisch contract.
- **`bereken_death_spiral`** — simuleert het vertrek van de goedkoopste klanten en het effect op het portfoliogemiddelde.
- **`bereken_gevoeligheid`** — voert de gevoeligheidsanalyse op prijsvolatiliteit uit.

In [ ]:
def laad_epex(bestand):
    """Laad een Ember-prijsbestand en behandel zomertijd-/wintertijdovergangen."""
    ember = pd.read_csv(bestand).rename(columns={
        "Datetime (Local)": "timestamp_lokaal",
        "Price (EUR/MWh)":  "Prijs_EUR_MWh",
    })
    ember["timestamp_lokaal"] = pd.to_datetime(ember["timestamp_lokaal"])
    ember["Prijs_EUR_MWh"]    = pd.to_numeric(ember["Prijs_EUR_MWh"], errors="coerce")
    ember["Prijs_EUR_KWh"]    = ember["Prijs_EUR_MWh"] / 1000

    # Extraheer maand/dag/uur voor de koppeling met de Fluvius-data
    ember["maand"] = ember["timestamp_lokaal"].dt.month
    ember["dag"]   = ember["timestamp_lokaal"].dt.day
    ember["uur"]   = ember["timestamp_lokaal"].dt.hour

    # Wintertijdovergang (oktober): uur 02:00 komt twee keer voor → middel
    ember = ember.groupby(["maand", "dag", "uur"]).agg(
        Prijs_EUR_MWh=("Prijs_EUR_MWh", "mean"),
        Prijs_EUR_KWh=("Prijs_EUR_KWh", "mean"),
    ).reset_index()

    # Zomertijdovergang (27 maart): uur 02:00 bestaat niet → interpoleer als ontbreekt
    if (3, 27, 2) not in set(zip(ember["maand"], ember["dag"], ember["uur"])):
        rij_voor = ember[(ember["maand"] == 3) & (ember["dag"] == 27) & (ember["uur"] == 1)]
        rij_na   = ember[(ember["maand"] == 3) & (ember["dag"] == 27) & (ember["uur"] == 3)]
        if len(rij_voor) > 0 and len(rij_na) > 0:
            mwh = (rij_voor["Prijs_EUR_MWh"].values[0] + rij_na["Prijs_EUR_MWh"].values[0]) / 2
            ember = pd.concat([ember, pd.DataFrame([{
                "maand": 3, "dag": 27, "uur": 2,
                "Prijs_EUR_MWh": mwh, "Prijs_EUR_KWh": mwh / 1000,
            }])], ignore_index=True)

    mediaan = ember["Prijs_EUR_MWh"].median()
    print(f"  {bestand}: {len(ember)} uurprijzen | mediaan {mediaan:.2f} EUR/MWh")
    return ember, mediaan


def bereken_kostenpositie_2400(ember, mediaan):
    """
    Bereken het individuele leverancierstarief per klant voor alle 2.400 Fluvius-meters
    en deel hen in kostenkwartielen.
    """
    # Laad alle 8 profielbestanden en koppel EPEX-prijzen
    delen = []
    for profiel_label, bestand in FLUVIUS_2400.items():
        df = pd.read_csv(bestand)

        # UTC → Belgische lokale tijd; NaT verwijderen (= 27 maart 02:00)
        df["Datum_Startuur"] = pd.to_datetime(
            df["Datum_Startuur"], utc=True
        ).dt.tz_convert("Europe/Brussels").dt.tz_localize(None)
        df = df.dropna(subset=["Datum_Startuur"]).copy()

        # Negatieve waarden (meetfouten) → 0
        df["Volume_Afname_KWh"]   = pd.to_numeric(df["Volume_Afname_KWh"],
                                                   errors="coerce").fillna(0).clip(lower=0)
        df["Volume_Injectie_KWh"] = pd.to_numeric(df["Volume_Injectie_KWh"],
                                                   errors="coerce").fillna(0).clip(lower=0)

        # Niet-PV klanten kunnen niet injecteren
        df.loc[df["PV_Installatie_Indicator"] == 0, "Volume_Injectie_KWh"] = 0.0

        df["maand"] = df["Datum_Startuur"].dt.month
        df["dag"]   = df["Datum_Startuur"].dt.day
        df["uur"]   = df["Datum_Startuur"].dt.hour
        df["Profiel_Label"] = profiel_label
        delen.append(df)

    data = pd.concat(delen, ignore_index=True)

    # Koppel EPEX op maand/dag/uur in lokale tijd
    data = data.merge(ember[["maand", "dag", "uur", "Prijs_EUR_KWh", "Prijs_EUR_MWh"]],
                      on=["maand", "dag", "uur"], how="left")
    data["Prijs_EUR_KWh"] = data["Prijs_EUR_KWh"].fillna(mediaan / 1000)
    data["Prijs_EUR_MWh"] = data["Prijs_EUR_MWh"].fillna(mediaan)

    # Bereken per meter het individuele leverancierstarief (volume-gewogen EPEX)
    records = []
    for (ean_id, profiel_label), kd in data.groupby(["EAN_ID", "Profiel_Label"]):
        tot_afn = kd["Volume_Afname_KWh"].sum()
        tot_inj = kd["Volume_Injectie_KWh"].sum()
        if tot_afn == 0:
            continue

        # Volume-gewogen gemiddelde EPEX op afname-uren + risicopremie
        gem_afn = (kd["Volume_Afname_KWh"] * kd["Prijs_EUR_KWh"]).sum() / tot_afn
        ind_tarief_afn = gem_afn * (1 + RISICO_PREMIE)

        if tot_inj > 0:
            gem_inj = (kd["Volume_Injectie_KWh"] * kd["Prijs_EUR_KWh"]).sum() / tot_inj
            ind_tarief_inj = gem_inj * (1 - RISICO_PREMIE)
        else:
            ind_tarief_inj = np.nan

        records.append({
            "EAN_ID": ean_id, "Profiel_Label": profiel_label,
            "Afname_KWh":          round(tot_afn, 2),
            "Injectie_KWh":        round(tot_inj, 2),
            "Ind_Tarief_Afname":   round(ind_tarief_afn, 6),
            "Ind_Tarief_Injectie": round(ind_tarief_inj, 6) if not np.isnan(ind_tarief_inj) else np.nan,
        })

    klant_df = pd.DataFrame(records)

    # Portfoliogemiddelde: volume-gewogen over alle 2.400 meters
    gem_portf_afn = ((klant_df["Afname_KWh"] * klant_df["Ind_Tarief_Afname"]).sum()
                     / klant_df["Afname_KWh"].sum())
    inj_sub = klant_df[klant_df["Injectie_KWh"] > 0].dropna(subset=["Ind_Tarief_Injectie"])
    gem_portf_inj = ((inj_sub["Injectie_KWh"] * inj_sub["Ind_Tarief_Injectie"]).sum()
                     / inj_sub["Injectie_KWh"].sum()) if len(inj_sub) > 0 else 0.0
    klant_df["Portfolio_Afname_excl_btw"] = round(gem_portf_afn, 6)
    klant_df["Portfolio_Injectie"]        = round(gem_portf_inj, 6)

    # Verdeling in kostenkwartielen op basis van individueel afnametarief
    klant_sorted = klant_df.sort_values("Ind_Tarief_Afname").reset_index(drop=True)
    N = len(klant_sorted)
    klant_sorted["Kwartiel"] = pd.cut(klant_sorted.index,
                                       bins=[0, N//4, N//2, 3*N//4, N],
                                       labels=KWARTIELEN, include_lowest=True)
    kwartiel_df = (pd.crosstab(klant_sorted["Kwartiel"], klant_sorted["Profiel_Label"],
                                normalize="index") * 100).reset_index()

    print(f"  Portfoliogem. afname (excl. btw): {gem_portf_afn*100:.4f} ct/kWh")
    return klant_df, kwartiel_df


def laad_fluvius(bestand, ember, mediaan, filter_29feb=False):
    """Laad een 1.000-steekproef en koppel EPEX-prijzen op maand/dag/uur."""
    fluvius = pd.read_csv(bestand)

    # UTC → Belgische lokale tijd; NaT van zomertijdovergang verwijderen
    fluvius["Datum_Startuur"] = pd.to_datetime(
        fluvius["Datum_Startuur"], utc=True
    ).dt.tz_convert("Europe/Brussels").dt.tz_localize(None)
    fluvius = fluvius.dropna(subset=["Datum_Startuur"]).copy()

    fluvius["Volume_Afname_KWh"]   = pd.to_numeric(fluvius["Volume_Afname_KWh"],
                                                    errors="coerce").fillna(0).clip(lower=0)
    fluvius["Volume_Injectie_KWh"] = pd.to_numeric(fluvius["Volume_Injectie_KWh"],
                                                    errors="coerce").fillna(0).clip(lower=0)
    fluvius.loc[fluvius["PV_Installatie_Indicator"] == 0, "Volume_Injectie_KWh"] = 0.0

    # Crisisscenario: 29 februari wegfilteren (bestaat niet in 2022)
    if filter_29feb:
        fluvius = fluvius[~((fluvius["Datum_Startuur"].dt.month == 2) &
                            (fluvius["Datum_Startuur"].dt.day == 29))].copy()

    fluvius["maand"] = fluvius["Datum_Startuur"].dt.month
    fluvius["dag"]   = fluvius["Datum_Startuur"].dt.day
    fluvius["uur"]   = fluvius["Datum_Startuur"].dt.hour

    data = fluvius.merge(ember[["maand", "dag", "uur", "Prijs_EUR_KWh", "Prijs_EUR_MWh"]],
                         on=["maand", "dag", "uur"], how="left")
    data["Prijs_EUR_KWh"] = data["Prijs_EUR_KWh"].fillna(mediaan / 1000)
    data["Prijs_EUR_MWh"] = data["Prijs_EUR_MWh"].fillna(mediaan)
    data["Profiel_Label"] = data["Profiel"].map(PROFIEL_LABELS).fillna(data["Profiel"])
    return data


def bereken_tarieven(data):
    """
    Bereken voor één scenario:
    - vast portfoliotarief (volume-gewogen EPEX × risicopremie)
    - individueel klanttarief
    - jaarkost op vast vs dynamisch contract per klant
    """
    tot_afname   = data["Volume_Afname_KWh"].sum()
    tot_injectie = data["Volume_Injectie_KWh"].sum()

    gem_afn  = (data["Volume_Afname_KWh"] * data["Prijs_EUR_KWh"]).sum() / tot_afname
    vast_afn = gem_afn * (1 + RISICO_PREMIE)

    if tot_injectie > 0:
        gem_inj  = (data["Volume_Injectie_KWh"] * data["Prijs_EUR_KWh"]).sum() / tot_injectie
        vast_inj = gem_inj * (1 - RISICO_PREMIE)
    else:
        vast_inj = 0.0

    records = []
    for (ean_id, profiel), kd in data.groupby(["EAN_ID", "Profiel"]):
        tot_afn_k = kd["Volume_Afname_KWh"].sum()
        tot_inj_k = kd["Volume_Injectie_KWh"].sum()
        if tot_afn_k == 0:
            continue

        # Individueel leverancierstarief (basis voor rangschikking in spiraal)
        gem_afn_k       = (kd["Volume_Afname_KWh"] * kd["Prijs_EUR_KWh"]).sum() / tot_afn_k
        ind_tarief_excl = gem_afn_k * (1 + RISICO_PREMIE)

        # Jaarkost op vast vs dynamisch contract (energiecomponent, excl. btw)
        kost_vast_afn  = tot_afn_k * vast_afn
        tarief_dyn_uur = (DYN_ALPHA * kd["Prijs_EUR_MWh"] + DYN_BETA) / 100
        kost_dyn_afn   = (kd["Volume_Afname_KWh"] * tarief_dyn_uur).sum()

        if tot_inj_k > 0:
            kost_vast_inj  = tot_inj_k * vast_inj
            vergoeding_dyn = (DYN_ALPHA_INJ * kd["Prijs_EUR_MWh"] - DYN_BETA_INJ) / 100
            kost_dyn_inj   = (kd["Volume_Injectie_KWh"] * vergoeding_dyn).sum()
        else:
            kost_vast_inj = kost_dyn_inj = 0.0

        records.append({
            "EAN_ID": ean_id, "Profiel": profiel,
            "Profiel_Label":         PROFIEL_LABELS.get(profiel, profiel),
            "Afname_KWh":            round(tot_afn_k, 2),
            "Injectie_KWh":          round(tot_inj_k, 2),
            "Ind_Tarief_Afname":     round(ind_tarief_excl, 6),
            "Portfolio_Afname_ct":   round(vast_afn * 100, 4),
            "Portfolio_Injectie_ct": round(vast_inj * 100, 4),
            "Kost_Vast_Afname":      round(kost_vast_afn, 2),
            "Kost_Dyn_Afname":       round(kost_dyn_afn, 2),
            "Opbr_Vast_Injectie":    round(kost_vast_inj, 2),
            "Opbr_Dyn_Injectie":     round(kost_dyn_inj, 2),
            "Dyn_Beter_Afname":      kost_dyn_afn < kost_vast_afn,
        })

    return vast_afn * 100, vast_inj * 100, pd.DataFrame(records)


def bereken_death_spiral(klant):
    """
    Simuleer het vertrek van de goedkoopste klanten in stappen van 2%.
    Bereken na elke stap het nieuwe portfoliogemiddelde van de achterblijvers.
    """
    klant_sorted = klant.sort_values("Ind_Tarief_Afname").reset_index(drop=True)
    N = len(klant_sorted)
    records = []

    for stap in range(SPIRAL_STAPPEN + 1):
        pct = stap / SPIRAL_STAPPEN
        achterblijvers = klant_sorted.iloc[int(N * pct):]
        if len(achterblijvers) < 10:
            break

        tot_afname = achterblijvers["Afname_KWh"].sum()
        if tot_afname == 0:
            break

        # Nieuw portfoliogemiddelde = volume-gewogen tarief van de achterblijvers
        gem_excl = ((achterblijvers["Ind_Tarief_Afname"] * achterblijvers["Afname_KWh"]).sum()
                    / tot_afname)
        records.append({
            "pct_vertrokken": round(pct * 100, 2),
            "energie_ct":     round(gem_excl * 100, 4),
        })

    return pd.DataFrame(records)


def bereken_profieldrempels(klant):
    """Bereken op welk punt in de portfolio 80% van elk profieltype vertrokken is."""
    klant_sorted = klant.sort_values("Ind_Tarief_Afname").reset_index(drop=True)
    N = len(klant_sorted)
    records = []
    for profiel, sub in klant_sorted.groupby("Profiel_Label"):
        if len(sub) == 0:
            continue
        idx = np.percentile(sub.index.to_numpy(), DREMPEL_PROFIEL * 100)
        records.append({
            "Profiel_Label": profiel,
            "pct_drempel":   round((idx + 1) / N * 100, 2),
        })
    return pd.DataFrame(records).sort_values("pct_drempel")


def bereken_gevoeligheid(data):
    """
    Bereken voor elk volatiliteitsscenario per profieltype het % klanten
    waarvoor dynamisch goedkoper is dan vast.

    Schaling: nieuwe_prijs = gemiddelde + (originele_prijs − gemiddelde) × factor.
    Het jaargemiddelde blijft daardoor gelijk; enkel de spreiding wijzigt.
    """
    gem_prijs = data["Prijs_EUR_MWh"].mean()
    records = []

    for vol_naam, vol_factor in VOLATILITEIT_SCENARIOS.items():
        data_vol = data.copy()
        data_vol["Prijs_EUR_MWh"] = gem_prijs + (data["Prijs_EUR_MWh"] - gem_prijs) * vol_factor
        data_vol["Prijs_EUR_KWh"] = data_vol["Prijs_EUR_MWh"] / 1000

        tot_afname = data_vol["Volume_Afname_KWh"].sum()
        gem_afn    = (data_vol["Volume_Afname_KWh"] * data_vol["Prijs_EUR_KWh"]).sum() / tot_afname
        vast_afn   = gem_afn * (1 + RISICO_PREMIE)

        klant_records = []
        for (ean_id, profiel), kd in data_vol.groupby(["EAN_ID", "Profiel"]):
            tot_afn_k = kd["Volume_Afname_KWh"].sum()
            if tot_afn_k == 0:
                continue
            kost_vast      = tot_afn_k * vast_afn
            tarief_dyn_uur = (DYN_ALPHA * kd["Prijs_EUR_MWh"] + DYN_BETA) / 100
            kost_dyn       = (kd["Volume_Afname_KWh"] * tarief_dyn_uur).sum()
            klant_records.append({
                "Profiel_Label":    PROFIEL_LABELS.get(profiel, profiel),
                "dyn_beter_afname": kost_dyn < kost_vast,
            })

        klant_df = pd.DataFrame(klant_records)
        for profiel in PROFIEL_VOLGORDE:
            sub = klant_df[klant_df["Profiel_Label"] == profiel]
            if len(sub) == 0:
                continue
            records.append({
                "volatiliteit":  vol_naam,
                "vol_factor":    vol_factor,
                "profiel":       profiel,
                "vast_ct_incl":  round(vast_afn * 100, 2),
                "pct_dyn_beter": round(sub["dyn_beter_afname"].mean() * 100, 1),
            })

    return pd.DataFrame(records)

### Hoofdprogramma

Voert de hele simulatie uit en slaat alle resultaten op als CSV-bestanden. Onderstaande cel kan een paar minuten duren afhankelijk van de rekenkracht.

**Outputbestanden:**

- `individuele_tarieven_2400.csv` — leverancierstarief per klant, alle 2.400 meters
- `kostenkwartielen_2400.csv` — profielverdeling per kostenkwartiel
- `tarieven_<scenario>.csv` — klantresultaten per scenario (4 bestanden)
- `spiral_<scenario>.csv` — death spiral curve per scenario (4 bestanden)
- `spiral_baseline_2022.csv` — death spiral curve crisisscenario
- `spiral_profieldrempels.csv` — wanneer 80% van elk profiel vertrokken is
- `dyn_vs_vast.csv` — % klanten waarvoor dynamisch beter is per profiel/scenario
- `gevoeligheid_dyn_vast.csv` — gevoeligheidsanalyse volatiliteit

In [ ]:
print("="*60)
print("SIMULATIE STARTEN")
print("="*60)

# Stap 4a — EPEX-prijzen laden
print("\n[1/4] EPEX-prijzen laden...")
ember_2024, mediaan_2024 = laad_epex("Belgium_2024_MWh.csv")
ember_2022, mediaan_2022 = laad_epex("Belgium_2022_MWh.csv")

# Stap 4b — Kostenpositie alle 2.400 meters
print("\n[2/4] Individuele kostenpositie (2.400 meters)...")
klant_2400, kwartiel_2400 = bereken_kostenpositie_2400(ember_2024, mediaan_2024)
klant_2400.to_csv("individuele_tarieven_2400.csv", index=False)
kwartiel_2400.to_csv("kostenkwartielen_2400.csv", index=False)

# Stap 4c — Per scenario: tarieven + death spiral + dyn/vast vergelijking
print("\n[3/4] Tarieven, death spiral en dyn/vast per scenario...")
klant_data, spiral_data, dyn_vast_res = {}, {}, []

for scenario, bestand in FLUVIUS_BESTANDEN.items():
    label = SCENARIO_LABELS[scenario]
    print(f"\n  → Scenario {label}")
    data = laad_fluvius(bestand, ember_2024, mediaan_2024)

    vast_ct, vast_inj_ct, klant_df = bereken_tarieven(data)
    klant_df["Scenario"] = label
    klant_df.to_csv(f"tarieven_{scenario}.csv", index=False)
    klant_data[scenario] = klant_df

    spiral = bereken_death_spiral(klant_df)
    spiral.to_csv(f"spiral_{scenario}.csv", index=False)
    spiral_data[scenario] = spiral
    print(f"    Vast tarief excl. btw : {vast_ct:.2f} ct/kWh")
    print(f"    Death spiral          : {spiral['energie_ct'].iloc[0]:.2f} → "
          f"{spiral['energie_ct'].iloc[-1]:.2f} ct/kWh")

    for profiel in PROFIEL_VOLGORDE:
        sub = klant_df[klant_df["Profiel_Label"] == profiel]
        if len(sub) > 0:
            dyn_vast_res.append({
                "scenario":      label,
                "profiel":       profiel,
                "vast_ct_incl":  round(vast_ct, 2),
                "pct_dyn_beter": round(sub["Dyn_Beter_Afname"].mean() * 100, 1),
            })

# Profieldrempels voor baseline plot
drempels = bereken_profieldrempels(klant_data["baseline"])
drempels.to_csv("spiral_profieldrempels.csv", index=False)

# Crisisscenario 2022: baseline-klantenmix met 2022-prijzen
print("\n  → Crisisscenario 2022")
data_crisis = laad_fluvius(FLUVIUS_BESTANDEN["baseline"], ember_2022, mediaan_2022,
                            filter_29feb=True)
_, _, klant_c = bereken_tarieven(data_crisis)
spiral_c = bereken_death_spiral(klant_c)
spiral_c.to_csv("spiral_baseline_2022.csv", index=False)
print(f"    Death spiral: {spiral_c['energie_ct'].iloc[0]:.2f} → "
      f"{spiral_c['energie_ct'].iloc[-1]:.2f} ct/kWh")

pd.DataFrame(dyn_vast_res).to_csv("dyn_vs_vast.csv", index=False)

# Stap 4d — Gevoeligheidsanalyse op de 2034-steekproef
print("\n[4/4] Gevoeligheidsanalyse volatiliteit (2034-steekproef)...")
data_2034    = laad_fluvius(FLUVIUS_BESTANDEN["2034"], ember_2024, mediaan_2024)
gevoeligheid = bereken_gevoeligheid(data_2034)
gevoeligheid.to_csv("gevoeligheid_dyn_vast.csv", index=False)

print("\n" + "="*60)
print("SAMENVATTING DEATH SPIRAL")
print("="*60)
print(f"\n{'Scenario':<30}{'Start':>10}{'Eind':>10}{'Stijging':>12}")
print("-"*60)
for scenario, spiral in spiral_data.items():
    s, e = spiral["energie_ct"].iloc[0], spiral["energie_ct"].iloc[-1]
    print(f"{SCENARIO_LABELS[scenario]:<30}{s:>10.2f}{e:>10.2f}{(e-s)/s*100:>+11.1f}%")
s, e = spiral_c["energie_ct"].iloc[0], spiral_c["energie_ct"].iloc[-1]
print(f"{'Crisisscenario 2022':<30}{s:>10.2f}{e:>10.2f}{(e-s)/s*100:>+11.1f}%")
print("\n✓ Alle outputbestanden aangemaakt.")